In [1]:
from uq_physicell.model_analysis import ModelAnalysisContext, run_simulations

db_path = "ObsData.db"  # Path to the database file
model_config = {"ini_path": "uq_config.ini", "struc_name": "ModelA"} # Example model configuration
sampler = 'Custom'  # Example sampler
params_info = { # Example parameters information
    'cell_cycle_entry': {'ref_value': 1440.0, 'perturbation': None},
    'apoptosis_rate': {'ref_value': 5.787e-05, 'perturbation': None}
}  
qoi_funcs = {
    'live_cell_count': lambda df: len( df[ df['dead'] == False ]),
    'integrated_dead_cell_count': lambda mcds_ts: sum(map(lambda mcds: len(mcds.get_cell_df()[mcds.get_cell_df()['dead'] == True]), mcds_ts))    
}
# Create Model Analysis Context
context = ModelAnalysisContext(db_path, model_config, sampler, params_info, qoi_funcs, num_workers=8)
# Sample parameter sets and run simulations
context.dic_samples = {0: {'cell_cycle_entry': 1440.0, 'apoptosis_rate': 5.787e-05}}
run_simulations(context)

Inserting parameter: cell_cycle_entry with properties: {'ref_value': 1440.0, 'perturbation': None}
Inserting parameter: apoptosis_rate with properties: {'ref_value': 5.787e-05, 'perturbation': None}
Inserting {'live_cell_count': "lambda df: len( df[ df['dead'] == False ])", 'integrated_dead_cell_count': "lambda mcds_ts: sum(map(lambda mcds: len(mcds.get_cell_df()[mcds.get_cell_df()['dead'] == True]), mcds_ts))"} QoIs into the database
Simulations completed and results stored in the database: ObsData.db.


In [2]:
import pandas as pd
import numpy as np
from uq_physicell.database.ma_db import load_output
# Load the simulation results and extract cell counts over time
df_qois_data = load_output(db_path)
df_all_data = pd.DataFrame()
for index, row in df_qois_data.iterrows():
    df_temp = row['Data']
    df_all_data = pd.concat([df_all_data, df_temp], ignore_index=True)

# Aggregate with multiple summary statistics
df_aggregated = df_all_data.groupby('time').agg({
    'live_cell_count': ['mean', 'std', 'min', 'max', 'count'],
    'integrated_dead_cell_count': ['mean', 'std', 'min', 'max', 'count']
}).reset_index()

# Flatten column names for easier access
df_aggregated.columns = ['_'.join(col).strip('_') if col[1] else col[0] 
                          for col in df_aggregated.columns.values]

time = df_aggregated['time'].values
live = df_aggregated['live_cell_count_mean'].values
dead = df_aggregated['integrated_dead_cell_count_mean'].values
df_aggregated.rename(columns={'time': 'Time', 'live_cell_count_mean': 'Live_Cells', 'integrated_dead_cell_count_mean': 'Sum_Dead_Cells'}, inplace=True)
display(df_aggregated)
# Add noise to the aggregated to columns Healthy Epithelial Cells and Infected Epithelial Cells, and save to CSV
columns = ['Live_Cells', 'Sum_Dead_Cells']
noise_std = 0.05  # Standard deviation of the noise (Normal distribution) no negative values
for col in columns:
    df_aggregated[col] = np.maximum(0, df_aggregated[col] + df_aggregated[col] * noise_std * np.random.randn(len(df_aggregated)))
df_aggregated.to_csv("ObsData.csv", index=False, columns=['Time']+columns)

,Time,Live_Cells,live_cell_count_std,live_cell_count_min,live_cell_count_max,live_cell_count_count,Sum_Dead_Cells,integrated_dead_cell_count_std,integrated_dead_cell_count_min,integrated_dead_cell_count_max,integrated_dead_cell_count_count
0,0.0,511.0,0.000000,511,511,5,NaN,NaN,NaN,NaN,0
1,240.0,580.8,8.786353,571,593,5,NaN,NaN,NaN,NaN,0
2,480.0,629.6,7.162402,622,641,5,NaN,NaN,NaN,NaN,0
3,720.0,665.0,9.924717,656,681,5,NaN,NaN,NaN,NaN,0
4,960.0,695.0,14.577380,685,720,5,NaN,NaN,NaN,NaN,0
5,1200.0,734.0,13.209845,720,753,5,NaN,NaN,NaN,NaN,0
6,1440.0,771.2,13.442470,755,787,5,NaN,NaN,NaN,NaN,0
7,1680.0,811.0,11.661904,800,830,5,NaN,NaN,NaN,NaN,0
8,1920.0,849.2,13.479614,833,864,5,NaN,NaN,NaN,NaN,0
9,2160.0,896.8,21.970435,866,923,5,NaN,NaN,NaN,NaN,0
